# 🎯 Module 15: MLflow & Experiment Tracking

> **Stage:** Machine Learning  
> **Level:** Beginner → Intermediate  
> **Module:** 15 — MLflow & Experiment Tracking  
> **Start Date:** March 2, 2026  
> **Previous:** 14 — ML Pipelines  
> **Next:** 16 — Model Deployment

---

Real ML projects involve **dozens or hundreds of experiments**. Without tracking, you end up with files like:

```text
model_final.pkl
model_final2.pkl
model_final_new.pkl
model_final_best.pkl
model_final_best_v2.pkl
```

You **don't remember** what changed between runs.

**MLflow** solves this — it tracks every experiment, its parameters, metrics, artifacts, and models in one place.

In this notebook we'll cover:

- What **experiment tracking** is
- **MLflow** overview
- **Experiments**, **Runs**, **Parameters**, **Metrics**, **Tags**, **Artifacts**, **Models**
- Installing & starting **MLflow UI**
- **Logging** parameters, metrics, tags, artifacts, models
- MLflow + **scikit-learn**
- MLflow + **Pipelines**
- MLflow + **hyperparameter tuning**
- **Autologging**
- **Model Registry** basics
- **Reproducibility**
- Common mistakes & best practices

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what **experiment tracking** is and why it matters
- Describe the **MLflow mental model** (Experiment → Run → Params / Metrics / Artifacts)
- Install & launch **MLflow UI**
- Log **parameters**, **metrics**, **tags**, **artifacts**, and **models**
- Use MLflow with **scikit-learn** and **Pipelines**
- Track **hyperparameter experiments**
- Use **autologging**
- Explain the difference between **Tracking** and the **Model Registry**
- Reason about **reproducibility**
- Follow MLflow **best practices**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# Check MLflow availability
try:
    import mlflow
    import mlflow.sklearn
    print('MLflow version:', mlflow.__version__)
except ImportError:
    print('MLflow not installed. Run: pip install mlflow')

---
## 🧠 1. What Is Experiment Tracking?

Recording information about **every ML experiment**:

```text
Experiment
    ↓
Model
    ↓
Parameters
    ↓
Metrics
    ↓
Artifacts
    ↓
Results
```

Instead of remembering:

```text
Model: Random Forest
n_estimators: 200
max_depth: 10
accuracy: 0.91
```

we track it **automatically**.

---
## 🎯 2. Why Experiment Tracking Matters

Imagine running 50 experiments:

```text
Experiment 1
Experiment 2
...
Experiment 50
```

### Without tracking

```text
model_final.pkl
model_final2.pkl
model_final_new.pkl
model_final_best.pkl
model_final_best_v2.pkl
```

You've forgotten what changed.

### With tracking

```text
Run 1  → params, metrics, artifacts
Run 2  → params, metrics, artifacts
Run 3  → params, metrics, artifacts
...
```

Everything is searchable, comparable, and reproducible.

---
## 🛠️ 3. What Is MLflow?

**MLflow** is an open-source platform for managing the ML lifecycle.

Main components:

```text
Tracking         ← we focus here
Projects
Models
Model Registry
```

We'll focus on **MLflow Tracking** — the part that records experiments.

---
## 🧠 4. MLflow Mental Model

```text
Experiment
    ↓
Runs
    ↓
Parameters / Metrics / Artifacts / Models
```

Example:

```text
Experiment: Customer Churn
       │
       ├── Run 1 (LogisticRegression)
       │    ├── Parameters
       │    ├── Metrics
       │    └── Model
       │
       ├── Run 2 (RandomForest)
       │
       └── Run 3 (XGBoost)
```

---
## 🗂️ 5. Experiment

A **logical container** for related ML runs.

```text
Experiment: Customer Churn Prediction
   ├── Run 1 → LogisticRegression
   ├── Run 2 → RandomForest
   ├── Run 3 → XGBoost
   └── Run 4 → Tuned XGBoost
```

All runs belong to the **same ML problem**.

---
## 🏃 6. Run

One **execution** of an ML experiment.

```text
Run 001
  Model: LogisticRegression
  C: 1.0
  Accuracy: 0.84
  F1: 0.82

Run 002
  Model: LogisticRegression
  C: 10
  Accuracy: 0.87
  F1: 0.85
```

Each run records **what happened** during that execution.

---
## 🔧 7. Parameters

**Configuration** values used during training.

Examples:

```text
learning_rate
max_depth
n_estimators
C
batch_size
epochs
```

```python
mlflow.log_param('max_depth', 10)
mlflow.log_params({'C': 1.0, 'max_iter': 1000})
```

---
## 📊 8. Metrics

Measurements of **model performance**.

Classification:
```text
accuracy
precision
recall
f1
roc_auc
```

Regression:
```text
MAE
MSE
RMSE
R²
```

```python
mlflow.log_metric('accuracy', 0.91)
mlflow.log_metrics({'accuracy': 0.91, 'f1': 0.89})
```

---
## ⚖️ 9. Parameters vs Metrics

| | Parameters | Metrics |
| --- | --- | --- |
| Meaning | configuration used | result achieved |
| Example | `max_depth=10` | `f1=0.91` |
| Answer | *What did I use?* | *What did I get?* |

---
## 🏷️ 10. Tags

Extra **metadata** about a run.

```text
model_type   = "random_forest"
dataset      = "customer_churn_v2"
environment  = "development"
developer    = "waleed"
```

```python
mlflow.set_tag('model_type', 'random_forest')
mlflow.set_tags({'dataset': 'churn_v2', 'env': 'dev'})
```

---
## 📦 11. Artifacts

**Files** produced during an ML run.

```text
model.pkl
confusion_matrix.png
feature_importance.csv
classification_report.txt
plots
```

```python
mlflow.log_artifact('confusion_matrix.png')
mlflow.log_artifacts('artifacts/')
```

---
## 🤖 12. Models

A trained model can be logged to MLflow.

```python
mlflow.sklearn.log_model(model, 'model')
```

Benefits:

- Store models with their runs
- Load models later
- Compare models across runs
- Register in **Model Registry**
- Deploy with standard tooling

---
## 🧠 13. Complete MLflow Mental Model

```text
Experiment
     ↓
   Run
     ↓
 ┌───┼────────┐
 ↓   ↓        ↓
Params Metrics Artifacts
              ↓
            Model
```

---
## 📥 14. Installing MLflow & Starting the UI

### Install
```bash
pip install mlflow
```

### Verify
```bash
mlflow --version
```

### Start UI
```bash
mlflow ui
```

Then open:

```text
http://127.0.0.1:5000
```

You can inspect **experiments, runs, params, metrics, artifacts, models**.

---
## 📝 15. Basic MLflow Tracking

```python
import mlflow

with mlflow.start_run():
    mlflow.log_param('model', 'LogisticRegression')
    mlflow.log_metric('accuracy', 0.87)
```

The run is automatically recorded.

In [ ]:
# Configure MLflow to write into a local folder inside the module
try:
    import mlflow
    BASE = Path('/Users/waleedkhokhar/Documents/AI Engineer/python-complete-journey/03 Machine-Learning/15-MLflow-and-Experiment-Tracking')
    BASE.mkdir(parents=True, exist_ok=True)
    tracking_dir = BASE / 'mlruns'
    tracking_dir.mkdir(exist_ok=True)
    mlflow.set_tracking_uri(f'file:{tracking_dir}')
    print('Tracking URI set to:', mlflow.get_tracking_uri())
except ImportError:
    print('MLflow not installed — pip install mlflow')

In [ ]:
# Basic MLflow run with params + metrics
try:
    import mlflow
    mlflow.set_experiment('Module15_Demo')

    with mlflow.start_run(run_name='basic_run'):
        mlflow.log_param('model', 'LogisticRegression')
        mlflow.log_params({'C': 1.0, 'max_iter': 1000})
        mlflow.log_metric('accuracy', 0.87)
        mlflow.log_metrics({'precision': 0.86, 'recall': 0.85, 'f1': 0.855})
        mlflow.set_tag('dataset', 'demo_v1')

    print('✅ Run logged. Start `mlflow ui` to view it.')
except ImportError:
    print('MLflow not installed — pip install mlflow')

---
## 📦 16. Load Dataset for Full Demo

In [ ]:
data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, ' Test:', X_test.shape)

---
## 🚀 17. MLflow + scikit-learn

Train → evaluate → log params → log metrics → log model.

In [ ]:
# Full training run tracked with MLflow
try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_experiment('Module15_Iris')

    with mlflow.start_run(run_name='LogReg_C1.0'):
        model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        f1  = f1_score(y_test, preds, average='weighted')

        mlflow.log_params({'model': 'LogisticRegression', 'C': 1.0, 'max_iter': 1000})
        mlflow.log_metrics({'accuracy': acc, 'f1': f1})
        mlflow.set_tags({'dataset': 'iris', 'task': 'classification'})
        mlflow.sklearn.log_model(model, 'model')

        print(f'Run logged — accuracy={acc:.4f}, f1={f1:.4f}')
except ImportError:
    print('MLflow not installed — pip install mlflow')

---
## 🎛️ 18. Tracking a Hyperparameter Sweep

Run several `C` values. Each becomes its **own run** in MLflow.

In [ ]:
try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_experiment('Module15_Iris_Sweep')

    for c in [0.01, 0.1, 1, 10]:
        with mlflow.start_run(run_name=f'C={c}'):
            model = LogisticRegression(C=c, max_iter=1000, random_state=42)
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)

            mlflow.log_param('C', c)
            mlflow.log_metric('accuracy', acc)

    print('✅ Sweep complete. Open MLflow UI to compare runs.')
except ImportError:
    print('MLflow not installed — pip install mlflow')

---
## 🏗️ 19. MLflow + Pipelines

Combine **Module 14** (pipelines) with **Module 15** (tracking).

In [ ]:
try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_experiment('Module15_Iris_Pipeline')

    with mlflow.start_run(run_name='Pipeline_LogReg'):
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=1000, random_state=42)),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        acc = accuracy_score(y_test, preds)

        mlflow.log_params({'model': 'LogReg', 'scaler': 'StandardScaler'})
        mlflow.log_metric('accuracy', acc)
        mlflow.sklearn.log_model(pipe, 'pipeline')

        print(f'Pipeline run logged — accuracy={acc:.4f}')
except ImportError:
    print('MLflow not installed — pip install mlflow')

---
## 🪄 20. Autologging

```python
mlflow.sklearn.autolog()
```

Automatically logs:

- parameters
- metrics
- model
- training info

In [ ]:
try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_experiment('Module15_Autolog')
    mlflow.sklearn.autolog(silent=True)

    with mlflow.start_run(run_name='RF_autolog'):
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train, y_train)

    print('✅ Autolog run completed.')
except ImportError:
    print('MLflow not installed — pip install mlflow')

---
## 📏 21. Custom Metrics

Log anything meaningful:

```text
business_score
training_time
inference_time
false_positive_rate
false_negative_rate
```

In [ ]:
import time

try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_experiment('Module15_CustomMetrics')

    with mlflow.start_run(run_name='with_timing'):
        t0 = time.time()
        model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
        model.fit(X_train, y_train)
        train_time = time.time() - t0

        mlflow.log_metric('training_time_sec', train_time)
        mlflow.log_metric('accuracy', accuracy_score(y_test, model.predict(X_test)))

    print(f'Custom metric logged: training_time_sec={train_time:.4f}')
except ImportError:
    print('MLflow not installed — pip install mlflow')

---
## 📦 22. Logging Artifacts

Save files and attach them to the run.

In [ ]:
try:
    import mlflow
    from sklearn.metrics import confusion_matrix

    mlflow.set_experiment('Module15_Artifacts')

    with mlflow.start_run(run_name='with_artifacts'):
        model = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
        preds = model.predict(X_test)
        cm = confusion_matrix(y_test, preds)

        # Save a plot
        fig, ax = plt.subplots(figsize=(4, 3))
        ax.imshow(cm, cmap='Blues')
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, cm[i, j], ha='center', va='center',
                        color='white' if cm[i, j] > cm.max() / 2 else 'black')
        ax.set_title('Confusion Matrix')
        plot_path = '/tmp/module15_confusion_matrix.png'
        fig.savefig(plot_path, dpi=80)
        plt.close(fig)

        # Save a report
        report_path = '/tmp/module15_report.txt'
        with open(report_path, 'w') as f:
            f.write(f'Accuracy: {accuracy_score(y_test, preds):.4f}\n')

        # Log both files
        mlflow.log_artifact(plot_path)
        mlflow.log_artifact(report_path)

    print('✅ Artifacts logged.')
except ImportError:
    print('MLflow not installed — pip install mlflow')

---
## 🗄️ 23. Model Registry — Overview

After training and tracking, models can be **registered**:

```text
Experiment → Run → Trained Model
                        ↓
              Registered Model
                        ↓
              Model Versions (v1, v2, v3)
                        ↓
                  Deployment
```

### Tracking vs Registry

| Tracking | Registry |
| --- | --- |
| What happened during training? | Which models are managed? |
| Runs, params, metrics | Versions, lifecycle |
| Experiment oriented | Deployment oriented |

---
## 🔁 24. Reproducibility

A reproducible experiment records:

```text
Model
Parameters
Metrics
Dataset version
Code version (Git commit)
Environment
Random seed
Artifacts
```

### MLflow + Git

```text
Git     →  code version
MLflow  →  experiment version
```

Together they give you a strong reproducibility workflow.

---
## ⚠️ 25. Common MLflow Mistakes

### 1. Tracking only accuracy
Log precision, recall, F1, ROC-AUC as appropriate.

### 2. Forgetting parameters
A metric alone can't be reproduced.

### 3. Not logging the model
Metrics without artifacts → harder to reuse.

### 4. Poor experiment naming
```text
test, test2, final, newfinal  ❌
Customer Churn Classification ✅
```

### 5. Ignoring dataset version
```python
mlflow.set_tag('dataset_version', 'v2')
```

### 6. Tracking without organization
Track what helps you **compare, reproduce, and operate** experiments.

---
## ✅ 26. Best Practices

1. Meaningful **experiment names**
2. Log important **parameters**
3. Log relevant **metrics**
4. Log **artifacts** (models, plots, reports)
5. Record **dataset info** as tags
6. Use **tags** for organization
7. Keep **code in Git**
8. Start with **local MLflow** before remote infrastructure

---
## 🏋️ 27. Hands-On Exercises

### Exercise 1 — First tracked run
Log a simple model's params and metrics with MLflow. Open the UI and inspect the run.

### Exercise 2 — Hyperparameter sweep
Run LogisticRegression for `C ∈ {0.01, 0.1, 1, 10}` and log each as a separate run. Compare in the UI.

### Exercise 3 — Model comparison
Run LogisticRegression, RandomForest, and SVM. Log each as a run. Which has the best F1?

### Exercise 4 — Pipeline logging
Wrap preprocessing + model in a `Pipeline` and log it with `mlflow.sklearn.log_model`.

### Exercise 5 — Autologging
Turn on `mlflow.sklearn.autolog()` and train a model. What does MLflow log automatically?

### Exercise 6 — Custom metrics
Log `training_time_sec` and `inference_time_sec` as custom metrics.

### Exercise 7 — Artifacts
Save a confusion matrix plot and a text report. Attach both as artifacts.

### Exercise 8 — Tags for organization
Add tags for `dataset_version`, `environment`, and `model_type`. Filter runs by tag in the UI.

---
## 🎤 28. Interview Cheat Sheet

| Question | Answer |
| --- | --- |
| What is MLflow? | An open-source platform for the ML lifecycle, including experiment tracking. |
| What is experiment tracking? | Recording information about ML training runs to understand and compare them. |
| What is a run? | One execution of an ML experiment. |
| What is an experiment? | A logical group of related runs. |
| Parameters vs metrics? | Parameters = configuration; Metrics = performance. |
| What are artifacts? | Files produced during a run. |
| What are tags? | Metadata used to organize runs. |
| Can MLflow log models? | Yes (`mlflow.sklearn.log_model`). |
| What is autologging? | Automatic logging of supported training info. |
| What is the Model Registry? | A system for managing model versions and lifecycle. |
| Is MLflow the whole MLOps stack? | No — it covers a core part of it. |
| MLflow + Git? | Git tracks code; MLflow tracks experiments. |

---
## 🏁 29. Key Takeaways

### You now understand

- What **experiment tracking** is
- What **MLflow** is and its core components
- **Experiments** and **runs**
- **Parameters**, **metrics**, **tags**, **artifacts**, **models**
- Installing MLflow and starting the **UI**
- Logging **parameters**, **metrics**, **tags**, **artifacts**, **models**
- MLflow with **scikit-learn**
- MLflow with **pipelines**
- Tracking **hyperparameter sweeps**
- **Autologging**
- **Model Registry** basics
- **Reproducibility** with MLflow + Git
- Best practices & common mistakes

### The core pattern

```text
Experiment
    ↓
Run
    ↓
Parameters / Metrics / Artifacts / Model
```

### 🧭 Golden Rule

> **Every ML experiment should be tracked — not just to remember what you did, but so you can reproduce, compare, and deploy with confidence.**

---

### 🔗 What's Next?

**Module 16 — Model Deployment**

We'll take a **saved pipeline** and turn it into a **FastAPI service**, containerize it with **Docker**, and make it **production-ready**.